# MTAM Reproduction — ZuCo Sentiment Analysis (sentence-level)

Runs the **forked** repo `parmisbathaeiyan/EEG_Language_Alignment`, branch `reproduction`,
instead of patching upstream code inside this notebook. Every change lives as a git
commit on the fork, so `git log upstream/main..reproduction` is the exact list of
deviations from the published code (printed in section 2).

Uses the **original ZuCo `.mat` data** from Drive. The sentiment-label CSV ships in the repo
(`preprocessed/ZuCo/sentiment_labels_clean.csv`). Model runs use the upstream loader; diagnostic
scripts may inspect the same raw files without changing the training path.

**Current step:** v14 found three exploratory 1-NN sentence-level setups worth checking, but also
exposed that the old seeded split depended on dictionary/filesystem order. V15 first verifies the
released cache against an independent raw-data reconstruction, then evaluates those three frozen
setups over 50 canonical, order-independent sentence splits with sentence-label permutation
controls. This comes before another **MLP-EEG** run (Table 1 target: F1 0.480, accuracy 0.499).

**Branch state (commits over upstream):**
- compat (modern PyTorch/Colab) + reporting (labeled confusion matrix, macro P/R/F1, JSON/PNG)
- bug fixes: best-val checkpoint, complete test loader, seeded RNGs, configurable early stopping
- methodology fixes: 80/10/10 split, optional oversampling, standard scaled-dot-product attention
- reproducibility fix: canonical order-independent splitting with exact sentence IDs saved
- EEG-only runs skip unrelated BERT construction and embedding work
- restored the released MLP class to the active training path

Data and results stay on Drive; `data/` is gitignored and never committed.

## 1. Mount Drive & configure paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verified against Drive on 2026-07-12. Folder contains results<SUBJ>_SR.mat files.
OG_ZUCO_SR_DIR = '/content/drive/MyDrive/Thesis/Data/zuco_og_raw'
RESULTS_ROOT   = '/content/drive/MyDrive/Thesis/Results/reproduce_EEG_Language_Alignment'
EEG_CACHE      = '/content/eeg_dict_cache.pkl'   # runtime cache; delete if source data changes

FORK_URL = 'https://github.com/parmisbathaeiyan/EEG_Language_Alignment.git'
BRANCH   = 'reproduction'

import os
assert os.path.isdir(OG_ZUCO_SR_DIR), f'SR .mat folder not found: {OG_ZUCO_SR_DIR}'
mats = sorted(f for f in os.listdir(OG_ZUCO_SR_DIR) if f.endswith('.mat'))
assert mats, f'No .mat files in {OG_ZUCO_SR_DIR}'
os.makedirs(RESULTS_ROOT, exist_ok=True)
print('Drive mounted. SR .mat files:', mats)

## 2. Clone the fork (reproduction branch)

In [ ]:
%cd /content
!rm -rf /content/EEG_Language_Alignment
!git clone --branch {BRANCH} {FORK_URL}
%cd /content/EEG_Language_Alignment
print('\nDeviations from upstream:')
!git remote add upstream https://github.com/Jason-Qiu/EEG_Language_Alignment.git 2>/dev/null; git fetch -q upstream
!git log --oneline upstream/main..{BRANCH}

## 3. Install dependencies
The repo's `requirements.txt` is frozen for Python 3.7 / CUDA 11.3. Install modern,
Colab-compatible versions instead.

In [ ]:
!pip install -q transformers==4.40.0
!pip install -q POT           # Python Optimal Transport (Wasserstein loss path)
!pip install -q scipy numpy pandas scikit-learn tqdm matplotlib seaborn
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 4. Prepare data
The upstream loader reads `data/SR/*.mat` and `data/sentiment_labels_clean.csv`.
Symlink the `.mat` files into `data/SR/`, copy the repo's label CSV, make output dirs.

In [ ]:
%cd /content/EEG_Language_Alignment
import os, shutil

os.makedirs('data/SR', exist_ok=True)
for d in ['lr_curves', 'pred_labels', 'baselines']:
    os.makedirs(d, exist_ok=True)

# Symlink the original .mat files into data/SR/ (no copy).
for f in os.listdir(OG_ZUCO_SR_DIR):
    if f.endswith('.mat'):
        dst = f'data/SR/{f}'
        if not os.path.exists(dst):
            os.symlink(os.path.join(OG_ZUCO_SR_DIR, f), dst)

# Label CSV ships in the repo.
shutil.copy('preprocessed/ZuCo/sentiment_labels_clean.csv', 'data/sentiment_labels_clean.csv')

print('data/SR:', sorted(os.listdir('data/SR')))
import pandas as pd
_df = pd.read_csv('data/sentiment_labels_clean.csv')
print('labels:', _df.shape, '| dist:', _df['sentiment_label'].value_counts().to_dict())

## 5. Run harness
Defines `run_experiment()` (run this cell once; it does not train anything by itself).
Logs stream to screen and to a `.txt`; a results `.json` and learning-curve `.png` are
written to Drive under `RESULTS_ROOT/<folder>/`. Future training runs save and evaluate both
the primary minimum-validation-loss checkpoint and a secondary maximum-validation-accuracy
checkpoint; both outcomes are retained, but test performance is never used to choose between them.

In [ ]:
import subprocess, os, datetime

def run_experiment(modality, loss, folder_name, *,
                   model='transformer', level='sentence',
                   num_layers=1, num_heads=5, batch_size=64,
                   epochs=200, warm_steps=2000, dropout=0.3,
                   mlp_hidden_sizes=(256, 128, 64),
                   mlp_implementation='released', mlp_bias=0,
                   optimizer_type='scheduled_adam', lr=1e-5, weight_decay=1e-2,
                   eps=1e-4, adam_betas=(0.9, 0.98),
                   patience=20, oversample=1,
                   sanity_overfit_per_class=0, suffix=''):
    ts = datetime.datetime.now().strftime('%Y%m%d%H%M%S')
    beta1, beta2 = adam_betas
    opt_tag = (f'{optimizer_type}_lr{lr:g}_wd{weight_decay:g}_eps{eps:g}'
               f'_betas{beta1:g}-{beta2:g}')
    if optimizer_type == 'scheduled_adam':
        opt_tag += f'_ws{warm_steps}'
    if model == 'MLP':
        hidden_tag = '-'.join(map(str, mlp_hidden_sizes))
        impl_tag = f'{mlp_implementation}_bias{mlp_bias}'
        run_name = f'{model}_{modality}_{level}_{loss}_h{hidden_tag}_{impl_tag}_b{batch_size}_{opt_tag}_p{patience}'
    else:
        run_name = f'{model}_{modality}_{level}_{loss}_L{num_layers}H{num_heads}b{batch_size}_{opt_tag}_p{patience}'
    if sanity_overfit_per_class:
        run_name += f'_sanity{sanity_overfit_per_class}pc'
    run_name += f'_{suffix}' if suffix else ''
    run_dir  = os.path.join(RESULTS_ROOT, folder_name)
    os.makedirs(os.path.join(run_dir, 'logs'),  exist_ok=True)
    os.makedirs(os.path.join(run_dir, 'plots'), exist_ok=True)
    log_path  = os.path.join(run_dir, 'logs',  f'{run_name}_{ts}.txt')
    json_path = os.path.join(run_dir,          f'{run_name}_{ts}.json')
    plot_dst  = os.path.join(run_dir, 'plots', f'{run_name}_{ts}.png')

    cmd = ['python', '-u', 'main_new.py',
           '--dataset', 'ZuCo', '--task', 'SA', '--level', level,
           '--modality', modality, '--model', model, '--loss', loss,
           '--batch_size', str(batch_size), '--epochs', str(epochs),
           '--num_layers', str(num_layers), '--num_heads', str(num_heads),
           '--dropout', str(dropout), '--warm_steps', str(warm_steps),
           '--mlp_hidden_sizes', *map(str, mlp_hidden_sizes),
           '--mlp_implementation', mlp_implementation, '--mlp_bias', str(mlp_bias),
           '--optimizer_type', optimizer_type, '--lr', str(lr),
           '--weight_decay', str(weight_decay), '--eps', str(eps),
           '--adam_beta1', str(beta1), '--adam_beta2', str(beta2),
           '--patience', str(patience), '--oversample', str(oversample),
           '--sanity_overfit_per_class', str(sanity_overfit_per_class),
           '--eeg_cache', EEG_CACHE,
           '--inference', '0', '--dev', '0', '--device', 'cuda',
           '--timestamp', ts, '--json_path', json_path, '--plot_dst', plot_dst]

    print(f'Run: {run_name} | {ts}\nDir: {run_dir}\n' + '-'*60)
    with open(log_path, 'w') as lf:
        lf.write(f'Run: {run_name}\nTimestamp: {ts}\nArgs: {" ".join(cmd[1:])}\n' + '-'*60 + '\n')
        p = subprocess.Popen(cmd, cwd='/content/EEG_Language_Alignment',
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1,
                             env={**os.environ, 'TQDM_DISABLE': '1',
                                  'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'})
        for line in p.stdout:
            print(line, end=''); lf.write(line); lf.flush()
        return_code = p.wait()
    if return_code != 0:
        raise RuntimeError(f'Training failed with exit code {return_code}. See {log_path}')
    print('-'*60 + f'\nDone. Log -> {log_path}')

print('run_experiment() ready.')

## 6. Run the diagnostic — v15 three-setup, 50-seed 1-NN audit

This no-training audit freezes the three promising v14 setups before examining more splits:

1. released cross-subject mean, then per-band/across-electrode Z-score, ordinary sentence 1-NN;
2. unaveraged records with the released per-record/per-band Z-score, same-participant 1-NN, then
   participant majority vote per sentence; and
3. unaveraged records with train-only participant/band pooled-all-electrode Z-score,
   different-participant 1-NN, then participant majority vote per sentence.

All 50 seeds use one shared order-independent 80/10/10 sentence splitter: IDs are sorted before
class-wise shuffling, and every participant recording of a sentence stays together. Exact IDs are
saved for every seed. For each setup the JSON includes validation/test accuracy, macro F1, confusion
matrices, distributions across seeds, paired differences from majority, and 200 sentence-label
permutations per seed. Permutations occur separately within train/validation/test so each split's
class counts stay fixed. The third setup is still **not LOSO**: the target participant contributes
other training sentences and its train-fitted normalization statistics.

Before scoring, the script builds `EEG_CACHE` if needed and compares every reconstructed feature and
label with it; it stops if the two are not equal within the declared tolerance. These are diagnostic
1-NN probes, not trained models or paper-comparable results. A GPU runtime is recommended. Run after
sections 1–4; section 5 is not needed.

In [ ]:
# No model training: 50 canonical split seeds x three frozen 1-NN setups.
import subprocess, os, datetime

audit_ts = datetime.datetime.now().strftime('%Y%m%d%H%M%S')
audit_dir = os.path.join(
    RESULTS_ROOT, 'v15_threeSetup_multiseedAudit'
)
os.makedirs(os.path.join(audit_dir, 'logs'), exist_ok=True)
audit_json = os.path.join(
    audit_dir, f'three_setup_1nn_50seeds_{audit_ts}.json'
)
audit_log = os.path.join(
    audit_dir, 'logs', f'three_setup_1nn_50seeds_{audit_ts}.txt'
)
audit_cmd = [
    'python', '-u', 'audit_multiseed_1nn.py',
    '--eeg_dir', 'data/SR',
    '--labels_csv', 'data/sentiment_labels_clean.csv',
    '--released_cache', EEG_CACHE,
    '--output_json', audit_json,
    '--seed_start', '0',
    '--num_seeds', '50',
    '--permutations', '200',
    '--permutation_seed', '20260719',
    '--device', 'auto',
]

print('Running v15 multi-seed 1-NN audit\nOutput:', audit_json)
with open(audit_log, 'w') as log_file:
    process = subprocess.Popen(
        audit_cmd, cwd='/content/EEG_Language_Alignment',
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
        env={**os.environ, 'TQDM_DISABLE': '1'},
    )
    for line in process.stdout:
        print(line, end='')
        log_file.write(line)
        log_file.flush()
    return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f'v15 multi-seed audit failed. See {audit_log}')
print('Audit complete. Log:', audit_log)

---
### After the diagnostic
Tell Codex the v15 50-seed audit has finished. We will inspect its Drive JSON for each setup's
validation/test distribution, confusion matrices, stability relative to majority, permutation-null
comparison, and validation–test relationship. Do not interpret these selected exploratory probes as
paper-comparable model scores. To pick up later commits, re-run section 2.